# Appendix II Figure 1: Panels A and B


In [12]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# Step 1. File names
# ============================================================

# Use the current working directory instead of __file__ for Jupyter environment
BASE_DIR = Path.cwd()

PANEL_A_INCOME_INPUT = BASE_DIR / "step2_income_category_average_shares.csv"
PANEL_B_INCOME_INPUT = BASE_DIR / "step4_income_seea_account_average_coverage.csv"
PANEL_A_COUNT_INPUT = BASE_DIR / "step1_indicator_category_summary.csv"

OUTPUT_PDF = BASE_DIR / "appendixIII_figure1_panels_A_B_income_coverage.pdf"
OUTPUT_SVG = BASE_DIR / "appendixIII_figure1_panels_A_B_income_coverage.svg"
OUTPUT_PNG = BASE_DIR / "appendixIII_figure1_panels_A_B_income_coverage.png"

# ============================================================
# Step 2. Figure settings from Step 6
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.unicode_minus"] = False

TITLE_FS = 20
AXIS_FS = 13
CELL_FS = 13
CBAR_FS = 13

# ============================================================
# Step 3. Income-group order
# ============================================================

INCOME_ORDER = [
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "High income",
    "Unclassified income",
]

# ============================================================
# Step 4. Panel A columns and labels
# ============================================================

PANEL_A_COLUMNS = [
    "Physical stock",
    "Monetary stock",
    "Physical flow",
    "Monetary flow",
    "Action_kept",
    "Policy Guide_kept",
    "HPS_monetary_only",
    "DDP_monetary_only",
    "MS-NMS_monetary_only",
    "ADE_monetary_only",
]

PANEL_A_BASE_LABELS = {
    "Physical stock": "Physical\nstock",
    "Monetary stock": "Monetary\nstock",
    "Physical flow": "Physical\nflow",
    "Monetary flow": "Monetary\nflow",
    "Action_kept": "Action",
    "Policy Guide_kept": "Policy\nguide",
    "HPS_monetary_only": "Monetary\nHPS",
    "DDP_monetary_only": "Monetary\nDDP",
    "MS-NMS_monetary_only": "Monetary\nMS-NMS",
    "ADE_monetary_only": "Monetary\nADE",
}

def make_panel_a_labels() -> dict[str, str]:
    counts = pd.read_csv(PANEL_A_COUNT_INPUT)

    required_cols = {"category", "indicator_count"}
    missing_cols = required_cols - set(counts.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_A_COUNT_INPUT.name} is missing required columns: {missing_cols}"
        )

    count_map = dict(
        zip(
            counts["category"].astype(str),
            counts["indicator_count"].astype(int),
        )
    )

    return {
        col: f"{PANEL_A_BASE_LABELS[col]}\n(n={count_map.get(col, 0)})"
        for col in PANEL_A_COLUMNS
    }

PANEL_A_LABELS = make_panel_a_labels()

# ============================================================
# Step 5. Panel B columns and labels
# ============================================================

PANEL_B_COLUMNS = [
    "Reported SEEA accounts",
    "Easily accessible accounts",
    "Found but not reported",
]

PANEL_B_LABELS = {
    "Reported SEEA accounts": "Reported\nSEEA accounts",
    "Easily accessible accounts": "Easily accessible\naccounts",
    "Found but not reported": "Account found but not\nreported",
}

# ============================================================
# Step 6. Helper functions
# ============================================================

def load_income_table(path: Path, value_cols: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path)

    required_cols = {"Income", "country_count", *value_cols}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f"{path.name} is missing required columns: {missing_cols}")

    df = df.copy()
    df["Income"] = df["Income"].fillna("Unclassified income")
    df["Income"] = df["Income"].replace({"nan": "Unclassified income", "": "Unclassified income"})

    ordered = (
        df.set_index("Income")
        .reindex(INCOME_ORDER)
        .dropna(how="all", subset=value_cols)
        .reset_index()
    )

    return ordered

def draw_heatmap(
    ax,
    df: pd.DataFrame,
    value_cols: list[str],
    label_map: dict[str, str],
    title: str,
    annotate_threshold: int = 15,
) -> mpl.image.AxesImage:
    values = df[value_cols].astype(float)

    image = ax.imshow(
        values,
        vmin=0,
        vmax=100,
        cmap="Blues",
        aspect="auto",
    )

    x_labels = [label_map[col] for col in value_cols]
    y_labels = [
        f"{income}\n(n={int(count)})"
        for income, count in zip(df["Income"], df["country_count"].fillna(0))
    ]

    ax.set_title(title, loc="left", fontsize=TITLE_FS, fontweight="bold", pad=10)

    ax.set_xticks(range(len(value_cols)))
    ax.set_xticklabels(x_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.tick_params(axis="both", length=0, pad=5)

    ax.set_xticks([x - 0.5 for x in range(1, len(value_cols))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(y_labels))], minor=True)
    ax.grid(which="minor", color="white", linewidth=0.9)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    for y in range(values.shape[0]):
        for x in range(values.shape[1]):
            if values.shape[1] > annotate_threshold:
                continue

            val = values.iat[y, x]
            color = "white" if val >= 55 else "#1f2933"

            ax.text(
                x,
                y,
                f"{val:.1f}",
                ha="center",
                va="center",
                fontsize=CELL_FS,
                fontweight="bold",
                color=color,
            )

    return image

def draw_panel_a(ax) -> mpl.image.AxesImage:
    income = load_income_table(PANEL_A_INCOME_INPUT, PANEL_A_COLUMNS)

    return draw_heatmap(
        ax,
        income,
        PANEL_A_COLUMNS,
        PANEL_A_LABELS,
        "A. Indicator coverage by income group across categories and boundaries",
        annotate_threshold=99,
    )

def draw_panel_b(ax) -> mpl.image.AxesImage:
    income = load_income_table(PANEL_B_INCOME_INPUT, PANEL_B_COLUMNS)

    return draw_heatmap(
        ax,
        income,
        PANEL_B_COLUMNS,
        PANEL_B_LABELS,
        "B. SEEA account coverage by income group",
    )

# ============================================================
# Step 7. Run and save appendix figure
# ============================================================

fig = plt.figure(figsize=(16.5, 11.8), constrained_layout=False)

outer = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[5, 4],
    hspace=0.42,
    top=0.955,
    bottom=0.075,
    left=0.115,
    right=0.890,
)

ax_a = fig.add_subplot(outer[0])
ax_b = fig.add_subplot(outer[1])

image = draw_panel_a(ax_a)
draw_panel_b(ax_b)

pos_a = ax_a.get_position()
pos_b = ax_b.get_position()

cax = fig.add_axes([0.915, pos_b.y0, 0.018, pos_a.y1 - pos_b.y0])
cbar = fig.colorbar(image, cax=cax)
cbar.ax.tick_params(labelsize=CBAR_FS, length=2)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

cbar.set_label(
    "Average country coverage (%)",
    fontsize=CBAR_FS,
    fontweight="bold",
)

fig.savefig(OUTPUT_PDF, bbox_inches="tight")
fig.savefig(OUTPUT_SVG, bbox_inches="tight")
fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")

plt.close(fig)

print("Appendix Panels A and B figure complete")
print(f"PDF: {OUTPUT_PDF}")
print(f"SVG: {OUTPUT_SVG}")
print(f"PNG: {OUTPUT_PNG}")

Appendix Panels A and B figure complete
PDF: D:\School\Fall 2024\DDL\ADB\00_Revision\Final Codes\Step 5\appendixIII_figure1_panels_A_B_income_coverage.pdf
SVG: D:\School\Fall 2024\DDL\ADB\00_Revision\Final Codes\Step 5\appendixIII_figure1_panels_A_B_income_coverage.svg
PNG: D:\School\Fall 2024\DDL\ADB\00_Revision\Final Codes\Step 5\appendixIII_figure1_panels_A_B_income_coverage.png
